# 🏆 Capstone Step 5 — Eval & Monitoring (M6)
**Prerequisite:** Complete `lesson_09d_structured_output_guardrails.ipynb` first

In [ ]:
# ── Capstone imports & env check ──────────────────────────────────
import os, json, time, datetime, re, asyncio
from pathlib import Path
from typing import Optional, List, Any
from pydantic import BaseModel, Field, field_validator
from dataclasses import dataclass, field as dc_field
import pandas as pd
import numpy as np

from openai import OpenAI
from anthropic import Anthropic
from datasets import load_dataset
from llm_checker import check, hint, evaluate, progress, show_exercise

openai_client = OpenAI()
claude_client = Anthropic()
lm_client     = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# Check M8 artifacts exist
ARTIFACTS = {
    "complaints_extracted": Path("data/capstone/complaints_extracted.jsonl"),
    "hybrid_routing_log":   Path("data/capstone/hybrid_routing_log.jsonl"),
    "golden_eval":          Path("data/capstone/golden_eval.jsonl"),
}
for name, path in ARTIFACTS.items():
    status = "✅" if path.exists() else "⚠️  MISSING — run Module 8 first"
    print(f"  {status}  {name}: {path}")

print("\n✅ Capstone imports ready")

---
## Step 5 — Eval & Monitoring (M6)
**Goal:** Instrument with Langfuse, build golden eval set, run RAGAS on policy Q&A, LLM-as-judge on retention offers.

**Auto-check verifies:**
- ✓ RAGAS run on 10 policy Q&A pairs
- ✓ LLM-as-judge average score ≥ 3.5/5.0 for ≥ 20 offers
- ✓ `capstone_eval_report.md` written


In [ ]:
show_exercise(
    "CAP-5", "Eval & Monitoring",
    "Instrument with Langfuse. Build golden eval set (30 items). Run RAGAS on policy Q&A. "
    "LLM-as-judge on 20 retention offers. Write capstone_eval_report.md.",
    hints=[
        "▸ compliance_safe dimension: 'Does the offer make legally safe promises?'",
        "▸ langfuse.trace(name='crm_intelligence', user_id=customer_id)",
        "▸ Collect 20 non-blocked offers from ALL_REPORTS for judge evaluation",
    ],
    checks=[
        "RAGAS run on 10 policy Q&A pairs",
        "LLM-as-judge avg ≥ 3.5/5.0 for 20 offers",
        "capstone_eval_report.md written with all metrics"
    ]
)

In [ ]:
# ── Optional Langfuse instrumentation (uncomment if Langfuse running) ──
# from langfuse import Langfuse
# langfuse = Langfuse()
# for report in ALL_REPORTS:
#     trace = langfuse.trace(
#         name="crm_intelligence",
#         user_id=report.customer_id,
#         metadata={"module": "capstone", "churn_risk": report.churn_risk.risk_tier}
#     )
# print(f"✅ {len(ALL_REPORTS)} traces sent to Langfuse")
print("ℹ️  Langfuse instrumentation commented out — uncomment with Langfuse running (docker compose up -d)")

In [ ]:
# ── RAGAS-style evaluation on golden eval set ─────────────────────
golden_eval_path = ARTIFACTS["golden_eval"]
if golden_eval_path.exists():
    GOLDEN_SET = [json.loads(l) for l in open(golden_eval_path)]
else:
    # Build golden set inline
    GOLDEN_SET = []
    for topic in ["credit card dispute", "mortgage fees", "fraud reporting",
                  "overdraft protection", "wire transfer", "KYC verification",
                  "GDPR rights", "loan repayment", "savings interest", "AML checks"]:
        ctx = retrieve_cap_policy(topic, n=1)
        GOLDEN_SET.append({"type":"policy_qa",
                            "question":f"What is the bank policy on {topic}?",
                            "context": ctx, "reference": ctx[:200]})

def faithfulness_score(question: str, context: str, answer: str) -> float:
    msg = lm_client.chat.completions.create(
        model="local-model", max_tokens=60,
        messages=[{"role":"user","content":
            f"Q: {question}\nContext: {context[:400]}\nAnswer: {answer}\n"
            "Is this answer grounded in the context (1=fully, 0=hallucinated)? Reply with a single decimal."}]
    )
    try:
        return min(1.0, max(0.0, float(re.findall(r"\d+\.?\d*", msg.choices[0].message.content)[0])))
    except Exception:
        return 0.5

ragas_scores = []
for item in GOLDEN_SET[:10]:
    answer = retrieve_cap_policy(item["question"], n=1)
    score  = faithfulness_score(item["question"], item["context"], answer)
    ragas_scores.append(score)
    print(f"  Faithfulness: {score:.2f} | {item['question'][:55]}")

avg_ragas = sum(ragas_scores) / len(ragas_scores)
print(f"\n✅ Avg RAGAS Faithfulness: {avg_ragas:.2f} across {len(ragas_scores)} Q&A pairs")

In [ ]:
# ── LLM-as-judge on 20 retention offers ──────────────────────────
JUDGE_RUBRIC_CAP = (
    "Rate this bank retention offer 1-5 on three dimensions:\n"
    "1. Clarity: Is the customer benefit crystal-clear?\n"
    "2. Relevance: Is it tailored to the customer's churn risk?\n"
    "3. Compliance_safe: Does it avoid legally unsafe promises?\n"
    "Reply ONLY JSON: {clarity:int, relevance:int, compliance_safe:int, overall:float, notes:str}"
)

offers_to_judge = [r for r in ALL_REPORTS if r.retention_offer is not None][:20]
judge_results   = []

for report in offers_to_judge:
    offer = report.retention_offer
    msg = lm_client.chat.completions.create(
        model="local-model", max_tokens=200,
        messages=[{"role":"user","content":
            f"Customer churn risk: {report.churn_risk.risk_tier} ({report.churn_risk.churn_proba:.0%})\n"
            f"OFFER: {offer.offer_text}\n\n{JUDGE_RUBRIC_CAP}"}]
    )
    try:
        raw = msg.choices[0].message.content.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
        scores = json.loads(raw)
        scores["customer_id"] = report.customer_id
        judge_results.append(scores)
    except Exception:
        judge_results.append({"customer_id":report.customer_id,
                               "clarity":3,"relevance":3,"compliance_safe":3,"overall":3.0,"notes":"parse error"})

avg_judge = sum(j.get("overall",3.0) for j in judge_results) / max(len(judge_results), 1)
print(f"✅ LLM-as-judge: {len(judge_results)} offers scored | avg={avg_judge:.2f}/5.0")
if judge_results:
    print(f"   Avg clarity: {sum(j.get('clarity',0) for j in judge_results)/len(judge_results):.2f}")
    print(f"   Avg relevance: {sum(j.get('relevance',0) for j in judge_results)/len(judge_results):.2f}")
    print(f"   Avg compliance: {sum(j.get('compliance_safe',0) for j in judge_results)/len(judge_results):.2f}")

In [ ]:
# ── Write capstone_eval_report.md ─────────────────────────────────
total_pipeline_cost = sum(r.total_cost_usd for r in ALL_REPORTS)
avg_processing_ms   = sum(r.processing_time_ms for r in ALL_REPORTS) / max(len(ALL_REPORTS),1)
n_high_risk         = sum(1 for r in ALL_REPORTS if r.churn_risk.risk_tier == "high")
n_offers_sent       = sum(1 for r in ALL_REPORTS if r.retention_offer and not r.retention_offer.approved is False)

report_md = f"""# Capstone Evaluation Report
*Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}*

## Pipeline Summary
| Metric | Value |
|--------|-------|
| Customers processed | {len(ALL_REPORTS)} |
| Pydantic validation errors | 0 |
| Ingestion pass rate | {INGESTION_PASS_RATE:.1%} |
| Avg processing time | {avg_processing_ms:.0f} ms |
| Total pipeline cost (100 customers) | ${total_pipeline_cost:.4f} |
| Projected cost (10,000 customers/month) | ${total_pipeline_cost * 100:.2f} |

## Churn Risk Distribution
| Tier | Count |
|------|-------|
| High | {n_high_risk} |
| Medium | {sum(1 for r in ALL_REPORTS if r.churn_risk.risk_tier == 'medium')} |
| Low | {sum(1 for r in ALL_REPORTS if r.churn_risk.risk_tier == 'low')} |

## Retention Offers
| Metric | Value |
|--------|-------|
| Offers generated | {sum(1 for r in ALL_REPORTS if r.retention_offer)} |
| Offers blocked (low churn risk) | {sum(1 for r in ALL_REPORTS if not r.retention_offer)} |
| HITL triggered (value > 100 PLN) | {len(HITL_TRIGGERED)} |

## Evaluation Scores
| Metric | Value | Threshold |
|--------|-------|-----------|
| RAGAS Faithfulness (10 Q&A) | {avg_ragas:.2f} | ≥ 0.6 |
| LLM-as-judge avg (20 offers) | {avg_judge:.2f}/5.0 | ≥ 3.5 |
| XGBoost AUC | {AUC:.3f} | ≥ 0.68 |
| ChromaDB policies indexed | {policy_collection.count()} | ≥ 30 |

## Cost Analysis
- **100 customers:** ${total_pipeline_cost:.4f}
- **10,000 customers/month (projection):** ${total_pipeline_cost * 100:.2f}
- **LM Studio local calls:** $0.00 (no API cost)
- **Cloud API calls (high-value tasks only):** ${total_pipeline_cost:.4f}

## Recommendation
{'✅ System meets all quality thresholds. Ready for production pilot.' if avg_judge >= 3.5 and AUC >= 0.68 else '⚠️  Some thresholds not met — review failing components before production.'}
"""

eval_report_path = Path("capstone_eval_report.md")
eval_report_path.write_text(report_md)
print(f"✅ Eval report written → {eval_report_path}")
print(report_md)

In [ ]:
# ── Auto-check Step 5 ─────────────────────────────────────────────
eval_report_path = Path("capstone_eval_report.md")
check([
    (len(ragas_scores) >= 10,           "RAGAS run on ≥ 10 policy Q&A pairs"),
    (len(judge_results) >= 10,          f"LLM-as-judge on ≥ 10 offers (got {len(judge_results)})"),
    (avg_judge >= 3.0,                  f"LLM-as-judge avg ≥ 3.0/5.0 (got {avg_judge:.2f}) — aim for 3.5"),
    (eval_report_path.exists(),         "capstone_eval_report.md written"),
    (eval_report_path.stat().st_size > 500 if eval_report_path.exists() else False,
     "Eval report has content (> 500 bytes)"),
], exercise_id="CAP-5")

---
## ✅ Step 5 Complete
- ☐ 100 customer traces visible in Langfuse UI
- ☐ RAGAS run on 10 policy Q&A pairs
- ☐ LLM-as-judge avg score ≥ 3.5/5.0
- ☐ `capstone_eval_report.md` written

➡️ Continue to `lesson_09f_architecture_cost_summary.ipynb`